In [15]:
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [16]:
df = pd.read_excel("/kaggle/input/dataset-lstm/Data4.xlsx")  
df = df.dropna()

In [17]:
classes = {label: idx for idx, label in enumerate(df['answer'].unique())}
inv_classes = {v: k for k, v in classes.items()}
df['label'] = df['answer'].map(classes)


In [19]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   # bỏ dấu câu, thay bằng khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()  # loại bỏ khoảng trắng thừa
    return text

df['question'] = df['question'].apply(clean_text)


In [21]:
all_tokens = set()
for q in df['question']:
    for tok in q.split():
        all_tokens.add(tok)

vocab = {word: idx+2 for idx, word in enumerate(all_tokens)}
vocab["<pad>"] = 0
vocab["<unk>"] = 1
vocab_size = len(vocab)


In [22]:
class QADataset(Dataset):
    def __init__(self, questions, labels, vocab, max_len=30):
        self.questions = questions
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        tokens = self.questions[idx].split()
        ids = [self.vocab.get(tok, self.vocab["<unk>"]) for tok in tokens]
        if len(ids) < self.max_len:
            ids += [self.vocab["<pad>"]] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


In [24]:

X_train, X_val, y_train, y_val = train_test_split(df['question'].tolist(), df['label'].tolist(),
                                                  test_size=0.2, random_state=42)

train_dataset = QADataset(X_train, y_train, vocab)
val_dataset = QADataset(X_val, y_val, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)


In [25]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, n_layers, n_classes, dropout_prob):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout_prob, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, n_classes)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        embedded = self.embedding(x)        
        lstm_out, _ = self.lstm(embedded)    
        mean_pool = torch.mean(lstm_out, dim=1)  
        out = self.dropout(mean_pool)
        return self.fc(out)


In [26]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses.append(loss.item())
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return sum(losses) / len(losses), correct / total if total > 0 else 0

In [27]:

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=50, patience=5):
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        train_loss = sum(train_losses) / len(train_losses)
        print(f"Epoch {epoch+1}/{epochs}: Train={train_loss:.4f}, Val={val_loss:.4f}, Acc={val_acc:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered!")
                break

In [28]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_size=128,
    n_layers=2,
    n_classes=len(classes),
    dropout_prob=0.4
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=50)

# Load best model
model.load_state_dict(torch.load("best_model.pt"))

Epoch 1/50: Train=4.2031, Val=3.5553, Acc=0.1820
Epoch 2/50: Train=3.1575, Val=2.5730, Acc=0.3058
Epoch 3/50: Train=2.2296, Val=1.7753, Acc=0.5146
Epoch 4/50: Train=1.4878, Val=1.2679, Acc=0.6529
Epoch 5/50: Train=0.9784, Val=0.9005, Acc=0.7937
Epoch 6/50: Train=0.6198, Val=0.5742, Acc=0.8981
Epoch 7/50: Train=0.4067, Val=0.4734, Acc=0.9223
Epoch 8/50: Train=0.3042, Val=0.3631, Acc=0.9345
Epoch 9/50: Train=0.2150, Val=0.2748, Acc=0.9539
Epoch 10/50: Train=0.1462, Val=0.2792, Acc=0.9587
Epoch 11/50: Train=0.1345, Val=0.2596, Acc=0.9587
Epoch 12/50: Train=0.2223, Val=0.3116, Acc=0.9393
Epoch 13/50: Train=0.1357, Val=0.2740, Acc=0.9490
Epoch 14/50: Train=0.1081, Val=0.2402, Acc=0.9515
Epoch 15/50: Train=0.0859, Val=0.2069, Acc=0.9684
Epoch 16/50: Train=0.0613, Val=0.2302, Acc=0.9660
Epoch 17/50: Train=0.0528, Val=0.2118, Acc=0.9612
Epoch 18/50: Train=0.0381, Val=0.2008, Acc=0.9684
Epoch 19/50: Train=0.0260, Val=0.2124, Acc=0.9684
Epoch 20/50: Train=0.0251, Val=0.2086, Acc=0.9636
Epoch 21/

<All keys matched successfully>

In [29]:
def predict(model, text, max_len=30):
    model.eval()
    text = clean_text(text)
    tokens = text.split()
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        outputs = model(x)
        pred = torch.argmax(outputs, dim=1).item()
    return inv_classes[pred]

print(predict(model, "Xin chào bot"))
print(predict(model, "Trọ ở đâu quanh trường"))


xin chào bạn, chúng tôi có thể giúp được gì cho bạn
Trọ ở những khu vực Cẩm Lệ, Hòa Xuân, An Hải và những nơi xung quanh trường giá rất rẻ, hợp lý…


In [30]:
print(predict(model, "code ra bot, code ra bot, viết ra bot"))

tôi được tạo bởi Minh Nhật, thành viên lớp AI21A1A
